In [30]:
import pandas as pd
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import parse_pubmed_rct_file

train_path = PROJECT_ROOT / "data/raw/pubmed_20k_rct/train.txt"
dev_path   = PROJECT_ROOT / "data/raw/pubmed_20k_rct/dev.txt"
test_path  = PROJECT_ROOT / "data/raw/pubmed_20k_rct/test.txt"

train_data = parse_pubmed_rct_file(train_path)
dev_data   = parse_pubmed_rct_file(dev_path)
test_data  = parse_pubmed_rct_file(test_path)

len(train_data), train_data.iloc[0].to_dict()


(180040,
 {'label': 'OBJECTIVE',
  'text': 'To investigate the efficacy of 6 weeks of daily low-dose oral prednisolone in improving pain , mobility , and systemic low-grade inflammation in the short term and whether the effect would be sustained at 12 weeks in older adults with moderate to severe knee osteoarthritis ( OA ) .'})

In [31]:
# Cell 2 — Quick sanity checks + label distribution
train_df = train_data
dev_df = dev_data
test_df = test_data

print("Train:", train_df.shape, "| Dev:", dev_df.shape, "| Test:", test_df.shape)
print("\nLabels (train):")
display(train_df["label"].value_counts())

Train: (180040, 2) | Dev: (30212, 2) | Test: (30135, 2)

Labels (train):


label
METHODS        59353
RESULTS        57953
CONCLUSIONS    27168
BACKGROUND     21727
OBJECTIVE      13839
Name: count, dtype: int64

In [32]:
# Cell 3 — Basic text-length EDA (tokens + characters)
train_df["char_len"] = train_df["text"].str.len()
train_df["word_len"] = train_df["text"].str.split().apply(len)

print("Character length summary:")
display(train_df["char_len"].describe())

print("\nWord length summary:")
display(train_df["word_len"].describe())


Character length summary:


count    180040.000000
mean        152.006276
std          79.000004
min           2.000000
25%          97.000000
50%         139.000000
75%         192.000000
max        1454.000000
Name: char_len, dtype: float64


Word length summary:


count    180040.000000
mean         26.338436
std          15.386976
min           1.000000
25%          16.000000
50%          23.000000
75%          33.000000
max         296.000000
Name: word_len, dtype: float64

In [33]:
# Cell 4 — Check class balance (percentages)
label_counts = train_df["label"].value_counts()
label_pct = (label_counts / label_counts.sum() * 100).round(2)

summary = pd.DataFrame({"count": label_counts, "pct_%": label_pct})
display(summary)


,count,pct_%
label,,
METHODS,59353,32.97
RESULTS,57953,32.19
CONCLUSIONS,27168,15.09
BACKGROUND,21727,12.07
OBJECTIVE,13839,7.69


In [34]:
# Cell 5 — Look at a few examples per label (quality check)
for lbl in train_df["label"].unique():
    print(f"\n=== {lbl} ===")
    sample = train_df[train_df["label"] == lbl].sample(3, random_state=42)["text"].tolist()
    for s in sample:
        print("-", s)



=== OBJECTIVE ===
- The purpose of this study was to characterize the relationship between heart rate and post-discharge outcomes in patients with hospitalization for heart failure ( HHF ) with reduced ejection fraction ( EF ) in sinus rhythm .
- It is unclear whether perchlorate exposure in early life affects neurodevelopment .
- The aim was to determine effectiveness of a self-management intervention in prevention of adverse outcomes ( catheter-related urinary tract infection , blockage , and accidental dislodgement ) .

=== METHODS ===
- We assessed safety in all patients who received at least one dose of alectinib .
- In the statistical analysis , it was differentiated between the control and treatment groups and between primi - and multiparous animals , and their interactions were analyzed .
- Intention-to-treat analysis will be used .

=== RESULTS ===
- Early complication rate was 5.1 % in USG group , 2.8 % in the IJB , and 0 % in the SCB ( P = 0.401 ) .
- The contrast medium in

In [35]:
# Cell 6 — Baseline split check: are labels consistent across splits?
labels_train = set(train_df["label"].unique())
labels_dev = set(dev_df["label"].unique())
labels_test = set(test_df["label"].unique())

print("Labels train:", labels_train)
print("Labels dev:  ", labels_dev)
print("Labels test: ", labels_test)
print("\nAll splits share same labels:", labels_train == labels_dev == labels_test)


Labels train: {'BACKGROUND', 'RESULTS', 'CONCLUSIONS', 'OBJECTIVE', 'METHODS'}
Labels dev:   {'RESULTS', 'METHODS', 'CONCLUSIONS', 'OBJECTIVE', 'BACKGROUND'}
Labels test:  {'RESULTS', 'METHODS', 'CONCLUSIONS', 'OBJECTIVE', 'BACKGROUND'}

All splits share same labels: True
